In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław'] 
kategorie = ['elektronika', 'odzież', 'żywność', 'książki'] 

def generate_transaction():
    # TWÓJ KOD
    # Zwróć słownik z polami: tx_id, user_id, amount, store, category, timestamp
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0,5000.0),2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

# TWÓJ KOD
# Pętla: generuj transakcję, wyślij do tematu 'transactions', wypisz, sleep 1s
for i in range(1000): #while true petla, zeby byla w nieskonczonosc, #ale wylacza sie przy wylaczeniu terminala czy cos 
    tx = generate_transaction() 
    producer.send('transactions', value=tx) 
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}") 
    time.sleep(1) #tu mozna uruchomic kilku producentow, zeby ta wysylka nie byla caly czas tak samo 

producer.flush()
producer.close() 

Overwriting producer.py


In [5]:
%%file consumer_filter.py 
from kafka import KafkaConsumer 
import json 

consumer = KafkaConsumer( 
    'transactions', #nazwa topic 
    bootstrap_servers='broker:9092', 
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) #deserializacja i loads, zamieniamy na slownik 
) 

print("Nasłuchuję na duże transakcje (amount > 3000)...")

#for message in consumer: #dostajemy sie do slownika 
#    print(message) #print.message 

     

# TWÓJ KOD 
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

for message in consumer: #dostajemy sie do slownika 
    val = message.value
    if val['amount']>3000:
        print(f"ALERT: {val['tx_id']} | {val['amount']} PLN | {val['store']} | {val['category']}")
    

Overwriting consumer_filter.py


In [7]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

consumer = KafkaConsumer( 
    'transactions', #nazwa topic 
    bootstrap_servers='broker:9092',
    group_id='consumer_enrich',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) #deserializacja i loads, zamieniamy na slownik 
) 

for message in consumer: #dostajemy sie do slownika 
    val = message.value
    if val['amount'] > 3000:
        val['risk_level'] = 'HIGH'
    elif val['amount'] > 1000:
        val['risk_level'] = 'MEDIUM'
    else:
        val['risk_level'] = 'LOW'

    print(val)        



Overwriting consumer_enrich.py


In [11]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'consumer_count',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. Zwiększ store_counts[store]
#   2. Dodaj amount do total_amount[store]
#   3. Co 10 wiadomości wypisz tabelę:
#      Sklep | Liczba | Suma | Średnia

for message in consumer:
    val = message.value
    store = val['store']
    amount = val['amount']
    
    store_counts[store] += 1
    
    if store not in total_amount:
        total_amount[store]=0.0

    total_amount[store] += amount

    msg_count += 1

    if msg_count % 10 == 0:
        for s in store_counts:
            liczba = store_counts[s]
            suma = total_amount[s]
            srednia = suma / liczba

            print(f"{s} | {liczba} | {suma:.2f} | {srednia:.2f}")
    

Overwriting consumer_count.py


In [12]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'consumer_stats',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

category_counts = Counter()
total_amount = {}
min_amount = {}
max_amount = {}
msg_count = 0

# TWÓJ KOD
for message in consumer:
    val = message.value
    category = val['category']
    amount = val['amount']
    
    category_counts[category] += 1
    
    if category not in total_amount:
        total_amount[category]=0.0
        min_amount[category] = amount
        max_amount[category] = amount

    total_amount[category] += amount
    min_amount[category] = min(min_amount[category], amount)
    max_amount[category] = max(max_amount[category], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        for c in category_counts:
            liczba = category_counts[c]
            suma = total_amount[c]
            minimum = min_amount[c]
            maximum = max_amount[c]

            print(f"{c} | {liczba} | {suma:.2f} | {minimum} | {maximum}")
    

Writing consumer_stats.py


In [ ]:
#zadanie 5.1 i 5.2

# Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?
# Konsument nie będzie wyświetlał żadnych danych, ponieważ nie będą one generowane w tym momencie przez 
# producenta.


# Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?
# Każda wiadomość trafi tylko do jednego z producentów.


# Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?
# W przetwarzaniu bezstanowym nie zachowujemy informacji o poprzednich transakcjach.
# W przetwarzaniu stanowym mamy kontekst historyczny w postaci np. wyliczonych statystyk.




In [16]:
%%file consumer_anomaly.py 
from kafka import KafkaConsumer 
import json 
from datetime import datetime

consumer = KafkaConsumer( 
    'transactions', #nazwa topic 
    bootstrap_servers='broker:9092', 
    group_id = 'consumer_anomaly',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) #deserializacja i loads, zamieniamy na slownik 
) 


# TWÓJ KOD 
# Napisz konsumenta wykrywającego anomalie prędkości: alert jeśli ten
# sam user_id wykona więcej niż 3 transakcje w ciągu 60 sekund.

user_history = {}

for message in consumer: 
    val = message.value
    user = val['user_id']
    transaction_time = datetime.fromisoformat(val['timestamp'])

    if user not in user_history:
        user_history[user] = []

    user_history[user].append(transaction_time)

    recent_transactions = []
    
    for saved_time in user_history[user]:
        time_diff = (transaction_time - saved_time).total_seconds()
        if time_diff <= 60.0:
            recent_transactions.append(saved_time)

    user_history[user] = recent_transactions

    if len(user_history[user]) > 3:
        print(f"ALERT: User {user} wykonał {len(user_history[user])} transakcji w ciągu ostatnich 60 sekund.")

    

Overwriting consumer_anomaly.py
